### Customer Complaints Data Cleaning.

In [8]:
%pip install openpyxl


   ---------------------------------------- 0.0/250.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/250.9 kB ? eta -:--:--
   - -------------------------------------- 10.2/250.9 kB ? eta -:--:--
   ---- ---------------------------------- 30.7/250.9 kB 435.7 kB/s eta 0:00:01
   ----------------- -------------------- 112.6/250.9 kB 819.2 kB/s eta 0:00:01
   ---------------------------------------  245.8/250.9 kB 1.5 MB/s eta 0:00:01
   ---------------------------------------- 250.9/250.9 kB 1.4 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
import numpy as np

In [9]:
customer_complaints = pd.read_excel('Consumer_Complaints.xlsx')

In [40]:
df = customer_complaints.copy()

In [41]:
df.head(15)

,Complaint ID,Submitted via,Date submitted,Date received,State,Product,Sub-product,Issue,Sub-issue,Company public response,Company response to consumer,Timely response?
0,4848023,Referral,2021-10-24,2021-10-27,NY,Mortgage,Conventional home mortgage,Applying for a mortgage or refinancing an exis...,NaN,Company has responded to the consumer and the ...,Closed with explanation,Yes
1,3621464,Web,2020-04-24,2020-04-24,FL,"Money transfer, virtual currency, or money ser...",Refund anticipation check,Lost or stolen check,NaN,Company has responded to the consumer and the ...,Closed with monetary relief,Yes
2,5818349,Web,2022-07-27,2022-07-27,CA,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Account information incorrect,Company has responded to the consumer and the ...,Closed with explanation,Yes
3,7233015,Referral,2023-07-10,2023-07-11,CA,Credit card or prepaid card,General-purpose prepaid card,Problem getting a card or closing an account,"Trouble getting, activating, or registering a ...",NaN,In progress,NaN
4,5820224,Referral,2022-07-27,2022-07-28,VA,Credit card or prepaid card,General-purpose credit card or charge card,Closing your account,Company closed your account,Company has responded to the consumer and the ...,Closed with explanation,Yes
5,6237961,Web,2022-11-23,2022-11-23,CA,Credit card or prepaid card,General-purpose credit card or charge card,Getting a credit card,Card opened as result of identity theft or fraud,Company has responded to the consumer and the ...,Closed with explanation,Yes
6,4348292,Web,2021-05-04,2021-05-04,TX,Checking or savings account,Checking account,Closing an account,Funds not received from closed account,Company has responded to the consumer and the ...,Closed with monetary relief,Yes
7,4090544,Web,2021-01-24,2021-01-24,CA,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Account status incorrect,Company has responded to the consumer and the ...,Closed with explanation,Yes
8,5337082,Web,2022-03-18,2022-03-18,TX,Credit card or prepaid card,General-purpose credit card or charge card,Fees or interest,Problem with fees,Company has responded to the consumer and the ...,Closed with monetary relief,Yes
9,6203657,Web,2022-11-14,2022-11-14,VA,Checking or savings account,Other banking product or service,Closing an account,Funds not received from closed account,Company has responded to the consumer and the ...,Closed with explanation,Yes


In [42]:
df.columns

Index(['Complaint ID', 'Submitted via', 'Date submitted', 'Date received',
       'State', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
       'Company public response', 'Company response to consumer',
       'Timely response?'],
      dtype='str')

In [72]:
rename_col = {'Complaint ID':'complaint_id',
              'Submitted via':'submitted_vis',
              'Date submitted':'submitted_date',
              'Date received':'received_date',
              'State':'state',
              'Product':'product',
              'Sub-product':'sub_product',
              'Issue':'issue',
              'Sub-issue':'sub_issue',
              'Company public response':'company_public_response',
              'Company response to consumer':'company_response',
              'Timely response':'timely_response'}

df.rename(columns=rename_col, inplace=True)

In [67]:
df.head(3)

,complaint_id,submitted_vis,submitted_date,received_date,state,product,sub_product,issue,sub_issue,company_public_response,company_response,Timely response?
0,4848023,Referral,2021-10-24,2021-10-27,NY,Mortgage,Conventional home mortgage,Applying for a mortgage or refinancing an exis...,Not_applicabel,Company has responded to the consumer and the ...,Closed with explanation,Yes
1,3621464,Web,2020-04-24,2020-04-24,FL,"Money transfer, virtual currency, or money ser...",Refund anticipation check,Lost or stolen check,Not_applicabel,Company has responded to the consumer and the ...,Closed with monetary relief,Yes
2,5818349,Web,2022-07-27,2022-07-27,CA,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Account information incorrect,Company has responded to the consumer and the ...,Closed with explanation,Yes


In [68]:
## Finding that complaint_id has all unique values or not,
print(df['complaint_id'].nunique(), ": It means complaints_id has all unique values")

62516 : It means complaints_id has all unique values


In [69]:
## Replacing NaN values in "sub_issue" with Not_applicabel,

df['sub_issue'] = df['sub_issue'].fillna('Not_applicabel')

In [70]:
## cheacking company_public_response,
df['company_public_response'] = df['company_public_response'].fillna('Not_recorded')

In [75]:
## overviewing submitted_via,
df['submitted_vis'] = df['submitted_vis'].replace({'Web Referral':'Referral',
                             'Email':'Other'})

In [76]:
df['submitted_vis'].value_counts()

submitted_vis
Web            45423
Referral       10856
Phone           4684
Postal mail     1318
Fax              233
Other              2
Name: count, dtype: int64

In [78]:
## Creating intake_lag_days column,
df['intake_lag_days'] = (df['received_date'] - df['submitted_date']).dt.days

In [79]:
# falgging lag more than 30 days,
df['lag_flag'] = df['intake_lag_days'] > 30

In [81]:
df.head(3)

,complaint_id,submitted_vis,submitted_date,received_date,state,product,sub_product,issue,sub_issue,company_public_response,company_response,Timely response?,intake_lag_days,lag_flag
0,4848023,Referral,2021-10-24,2021-10-27,NY,Mortgage,Conventional home mortgage,Applying for a mortgage or refinancing an exis...,Not_applicabel,Company has responded to the consumer and the ...,Closed with explanation,Yes,3,False
1,3621464,Web,2020-04-24,2020-04-24,FL,"Money transfer, virtual currency, or money ser...",Refund anticipation check,Lost or stolen check,Not_applicabel,Company has responded to the consumer and the ...,Closed with monetary relief,Yes,0,False
2,5818349,Web,2022-07-27,2022-07-27,CA,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Account information incorrect,Company has responded to the consumer and the ...,Closed with explanation,Yes,0,False


In [84]:
# overviewing sub_products,

df['sub_product'] = df['sub_product'].fillna('Not specified')